# 30 · 점수 합치기 — RRF vs Min-Max (KLUE-MRC)

강의 : 「스코어 병합 문제」 · 「RRF - 역수 순위 융합」 · 「Min-Max 정규화 퓨전」 · 「하이브리드 검색」

4-1 과 같은 평가 : 질문 500개, 정답 지문이 상위 10위 안이면 Hit, MRR = 정답 순위 역수의 평균.
두 검색 결과(각 상위 50개)를 한 번 뽑아 두고, **합치는 방법만** 바꿔 가며 비교한다.

| 이름 | 방법 |
|---|---|
| `bm25` | 4-1 의 `bm25_search()` |
| `vec` | 4-1 실습5 와 같음 : emb128 HNSW 로 후보 200 → 원본 4096 으로 재정렬 |

| 셀 | 하는 일 |
|---|---|
| 1 | 설정 |
| 2 | 연결과 평가 도구 |
| 3 | 두 검색 결과 뽑기 (질문마다 상위 50) |
| 4 | 단일 방법의 성적 |
| 5 | 왜 그냥 더하면 안 되나 — 척도 |
| 6 | RRF 가중치 곡선 |
| 7 | RRF — k 와 가중치 |
| 8 | Min-Max — α |
| 9 | Min-Max 의 약점 |
| 10 | 종합 |

**Run → Run All Cells** (셀 3 약 20초)

### 셀 1 · 설정
- `CAND` : 벡터 검색에서 emb128 로 추릴 후보 수 · `TOP` : 합치기 전에 각 방법이 내놓는 결과 수
- `MM_ALPHAS` : 셀 8(곡선 위 표시) · 셀 9(약점 예시의 정답 순위) · 셀 10(종합표)에 따로 올릴 Min-Max α

In [ ]:
# 셀 1 · 설정
DB_URL = "postgresql://lab:lab@db:5432/lab"
EVAL_N = 500     # 평가 질문 수 (4-1 과 같은 질문)
CAND   = 200     # 벡터 : emb128 HNSW 후보 수 → 원본 4096 재정렬
TOP    = 50      # 각 방법의 결과 수
MM_ALPHAS = [0.4, 0.6, 0.8]   # 셀 8 · 9 · 10 에서 따로 비교할 Min-Max α (BM25 비중)

### 셀 2 · 연결과 평가 도구
- `evaluate(순위목록)` : 질문 id → 지문 id 목록(점수 높은 순)을 받아 Hit@10 · MRR

In [ ]:
# 셀 2 · 연결과 평가 도구
from labutil import *
import pickle

conn = connect(DB_URL)


def evaluate(ranked, k=10):
    """ranked: 질문 id → [지문 id, ...] (점수 높은 순). 반환: Hit@10, MRR"""
    hit, rr = [], []
    for qid in QIDS:
        lst = list(ranked[qid])[:k]
        g = runs["gold"][qid]
        r = lst.index(g) + 1 if g in lst else None
        hit.append(r is not None); rr.append(1 / r if r else 0)
    return {"Hit@10": round(float(np.mean(hit)), 3), "MRR": round(float(np.mean(rr)), 3)}

### 셀 3 · 두 검색 결과 뽑기
질문마다 BM25 상위 50 과 벡터 상위 50 을 (지문 id, 점수) 로 받아 `results/klue_runs.pkl` 에 저장한다(31번이 재사용).
- 벡터 점수 = 코사인 유사도 `1 − (a <=> b)`

In [ ]:
# 셀 3 · 두 검색 결과 뽑기
os.makedirs("results", exist_ok=True)
conn.execute(f"SET hnsw.ef_search = {CAND}")          # 후보 CAND 개를 받으려면 ef_search ≥ CAND
with timer("검색 500 × 2"):
    rows = conn.execute(f"""
      SELECT e.id, e.doc_id, e.question, b.ids, b.sc, v.ids, v.sc
      FROM (SELECT * FROM questions WHERE id % 11 = 0 ORDER BY id LIMIT {EVAL_N}) e
      CROSS JOIN LATERAL (SELECT array_agg(doc_id ORDER BY score DESC) ids, array_agg(score ORDER BY score DESC) sc
                          FROM bm25_search(e.question, {TOP})) b
      CROSS JOIN LATERAL (SELECT array_agg(id ORDER BY sim DESC) ids, array_agg(sim ORDER BY sim DESC) sc FROM (
                            SELECT c.id, 1 - (c.embedding <=> e.embedding) AS sim
                            FROM (SELECT id, embedding FROM docs ORDER BY emb128 <=> e.emb128 LIMIT {CAND}) c
                            ORDER BY c.embedding <=> e.embedding LIMIT {TOP}) x) v""").fetchall()
conn.execute("RESET hnsw.ef_search")
runs = {"gold": {}, "question": {}, "bm25": {}, "vec": {}}
for qid, gold, text, bi, bs, vi, vs in rows:
    runs["gold"][qid], runs["question"][qid] = gold, text
    runs["bm25"][qid] = list(zip(bi or [], bs or []))
    runs["vec"][qid] = list(zip(vi, vs))
pickle.dump(runs, open("results/klue_runs.pkl", "wb"))
QIDS = sorted(runs["gold"])
print(len(QIDS), "질문 · BM25 결과가 50개 미만인 질문", sum(len(runs["bm25"][q_]) < TOP for q_ in QIDS))

### 셀 4 · 단일 방법의 성적

In [ ]:
# 셀 4 · 단일 방법의 성적
single = {m: evaluate({qid: [d for d, _ in runs[m][qid]] for qid in QIDS}) for m in ["bm25", "vec"]}
S = pd.DataFrame(single).T
fig, ax = plt.subplots(figsize=(7, 3.4))
x = np.arange(len(S))
ax.bar(x - 0.2, S["Hit@10"], 0.4, color="#3D484A", label="Hit@10"); ax.bar(x + 0.2, S["MRR"], 0.4, color="#FF7043", label="MRR")
for i, (h, m) in enumerate(S.values):
    ax.text(i - 0.2, h, f"{h:.3f}", ha="center", va="bottom"); ax.text(i + 0.2, m, f"{m:.3f}", ha="center", va="bottom")
ax.set_xticks(x, ["BM25", "벡터 (Qwen3)"]); ax.set_ylim(0, 1.1); ax.legend(); ax.set_title(f"단일 검색 (질문 {len(QIDS)}개)")
plt.show()
both = sum(runs["gold"][q_] in [d for d, _ in runs["bm25"][q_][:10]] and runs["gold"][q_] in [d for d, _ in runs["vec"][q_][:10]] for q_ in QIDS)
only_b = sum(runs["gold"][q_] in [d for d, _ in runs["bm25"][q_][:10]] for q_ in QIDS) - both
only_v = sum(runs["gold"][q_] in [d for d, _ in runs["vec"][q_][:10]] for q_ in QIDS) - both
print(f"상위 10 안에 정답 : 둘 다 {both} · BM25 만 {only_b} · 벡터만 {only_v} · 둘 다 놓침 {len(QIDS) - both - only_b - only_v}")

### 셀 5 · 왜 그냥 더하면 안 되나 — 척도가 다르다
- 왼쪽 두 그림 : 상위 10 의 점수 분포. BM25 는 수~수십, 코사인은 0~1
- 오른쪽 : 질문 하나의 후보를 한 평면에. 한쪽 목록에만 있는 지문은 다른 쪽 점수가 없다

In [ ]:
# 셀 5 · 척도 비교
fig, axes = plt.subplots(1, 3, figsize=(17, 4))
b = np.concatenate([[s for _, s in runs["bm25"][qid][:10]] for qid in QIDS])
v = np.concatenate([[s for _, s in runs["vec"][qid][:10]] for qid in QIDS])
axes[0].hist(b, bins=60, color="#3D484A"); axes[0].set_title(f"BM25 점수 (상위 10): {b.min():.0f} ~ {b.max():.0f}")
axes[1].hist(v, bins=60, color="#FF7043"); axes[1].set_title(f"코사인 유사도 (상위 10): {v.min():.2f} ~ {v.max():.2f}")
qid = next(i for i in QIDS if runs["gold"][i] in dict(runs["bm25"][i]) and runs["gold"][i] in dict(runs["vec"][i])
           and [d for d, _ in runs["vec"][i]].index(runs["gold"][i]) > 0)
bd, vd = dict(runs["bm25"][qid]), dict(runs["vec"][qid])
v_floor = min(vd.values()) - 0.03
both_ = [d for d in bd if d in vd]
axes[2].scatter([bd[d] for d in both_], [vd[d] for d in both_], s=22, color="#3D484A", label="양쪽 목록에 있음")
axes[2].scatter([bd[d] for d in bd if d not in vd], [v_floor] * sum(d not in vd for d in bd), s=22, marker="|", color="#FF7043", label="BM25 목록에만")
axes[2].scatter([0] * sum(d not in bd for d in vd), [vd[d] for d in vd if d not in bd], s=22, marker="_", color="#009CA6", label="벡터 목록에만")
g = runs["gold"][qid]
axes[2].scatter([bd[g]], [vd[g]], s=220, marker="*", color="#F6BD60", edgecolors="black", zorder=5, label="정답 지문")
axes[2].set_xlabel("BM25 점수"); axes[2].set_ylabel("코사인 유사도"); axes[2].legend(fontsize=7)
axes[2].set_title("질문 하나의 후보")
plt.tight_layout(); plt.show()
print("질문:", runs["question"][qid])

### 셀 6 · RRF 가중치 곡선
`RRF(d) = Σ 1 / (k + rank)`. 강의는 "k 로 변곡점을 조절"이라 했다 — 곡선과 그 감소량을 그려 본다.

In [ ]:
# 셀 6 · RRF 가중치 곡선
ranks = np.arange(1, 51)
fig, axes = plt.subplots(1, 2, figsize=(14, 3.8))
for k, c in zip([0, 10, 60, 200], ["#FF7043", "#009CA6", "#3D484A", "#9aa5a8"]):
    w = 1 / (k + ranks)
    axes[0].plot(ranks, w / w[0], color=c, label=f"k={k}")
    axes[1].plot(ranks[:-1], np.diff(w / w[0]), color=c, label=f"k={k}")
axes[0].set_xlabel("순위"); axes[0].set_ylabel("1위 대비 가중치"); axes[0].legend(); axes[0].set_title("1/(k+rank) — k 가 작을수록 앞쪽에 쏠림")
axes[1].set_xlabel("순위"); axes[1].set_ylabel("다음 순위로 갈 때 감소량"); axes[1].set_title("감소량이 단조롭게 줄어든다 = 변곡점 없음")
plt.tight_layout(); plt.show()
print("10위의 가중치 / 1위의 가중치 :", {k: round((1 / (k + 10)) / (1 / (k + 1)), 2) for k in [0, 10, 60, 200]})

### 셀 7 · RRF — k 와 가중치
- 왼쪽 : k 를 바꿔 BM25 + 벡터를 합침. 점선 = BM25 단독
- 오른쪽 : k = 60 에서 BM25 쪽에 가중치(벡터 = 1)

In [ ]:
# 셀 7 · RRF
def rrf(lists, k=60, w=None):
    w = w or [1] * len(lists)
    out = {}
    for qid in QIDS:
        s = {}
        for L, wi in zip(lists, w):
            for r, (d, _) in enumerate(runs[L][qid], 1):
                s[d] = s.get(d, 0) + wi / (k + r)
        out[qid] = sorted(s, key=s.get, reverse=True)
    return out


rrf_res = pd.DataFrame([{"k": k, **evaluate(rrf(["bm25", "vec"], k))} for k in [1, 5, 10, 20, 40, 60, 100, 200]])
wrrf = pd.DataFrame([{"BM25 가중치": w, **evaluate(rrf(["bm25", "vec"], 60, [w, 1]))} for w in [0.5, 1, 1.5, 2, 3, 5]])
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
for col, c in [("Hit@10", "#3D484A"), ("MRR", "#FF7043")]:
    axes[0].plot(rrf_res["k"], rrf_res[col], "-o", color=c, label=f"RRF {col}")
    axes[0].axhline(single["bm25"][col], color=c, ls=":", label=f"BM25 단독 {col}")
    axes[1].plot(wrrf["BM25 가중치"], wrrf[col], "-o", color=c, label=col)
axes[0].set_xscale("log"); axes[0].set_xlabel("RRF 상수 k"); axes[0].legend(fontsize=8); axes[0].set_title("k 를 바꾸면")
axes[1].set_xlabel("BM25 쪽 가중치 (벡터 = 1, k = 60)"); axes[1].legend(); axes[1].set_title("가중 RRF")
plt.tight_layout(); plt.show()
pd.concat({"k": rrf_res.set_index("k"), "BM25 가중치 (k=60)": wrrf.set_index("BM25 가중치")}).T

### 셀 8 · Min-Max — α
질문마다 각 목록 안에서 `(s − min) / (max − min)` 로 0~1 에 펴고 `α·BM25 + (1−α)·벡터`. 한쪽에만 있는 지문은 다른 쪽을 0 으로.
- 세로 점선 : `MM_ALPHAS` (0.4 · 0.6 · 0.8). 아래 표에서 ★

In [ ]:
# 셀 8 · Min-Max
def minmax_fuse(a, b, alpha):
    out = {}
    for qid in QIDS:
        fused = {}
        for L, wgt in [(a, alpha), (b, 1 - alpha)]:
            sc = np.array([s for _, s in runs[L][qid]], dtype=float)
            if len(sc) == 0:
                continue
            lo, hi = sc.min(), sc.max()
            for d, s in runs[L][qid]:
                fused[d] = fused.get(d, 0) + wgt * ((s - lo) / (hi - lo) if hi > lo else 1)
        out[qid] = sorted(fused, key=fused.get, reverse=True)
    return out


mm = pd.DataFrame([{"α (BM25 비중)": a, **evaluate(minmax_fuse("bm25", "vec", a))} for a in np.round(np.linspace(0, 1, 11), 1)])
best_rrf = rrf_res.loc[rrf_res["MRR"].idxmax()]
fig, ax = plt.subplots(figsize=(10, 4))
for col, c in [("Hit@10", "#3D484A"), ("MRR", "#FF7043")]:
    ax.plot(mm["α (BM25 비중)"], mm[col], "-o", color=c, label=f"Min-Max {col}")
    ax.axhline(best_rrf[col], color=c, ls="--", alpha=0.6, label=f"RRF 최고 {col} (k={int(best_rrf['k'])})")
for a in MM_ALPHAS:
    r = mm[np.isclose(mm["α (BM25 비중)"], a)].iloc[0]
    ax.axvline(a, color="#9aa5a8", ls=":", lw=1)
    ax.annotate(f"α={a}\nHit {r['Hit@10']:.3f}\nMRR {r['MRR']:.3f}", (a, r["MRR"]), textcoords="offset points",
                xytext=(0, -46), ha="center", fontsize=8, bbox=dict(boxstyle="round,pad=0.2", fc="white", ec="none", alpha=0.85))
ax.set_xlabel("α : 0 = 벡터만, 1 = BM25 만"); ax.legend(fontsize=8, ncol=2); ax.set_title("Min-Max 가중 평균")
plt.show()
T8 = mm.set_index("α (BM25 비중)").T
T8.columns = [f"{a}★" if np.isclose(a, MM_ALPHAS).any() else a for a in T8.columns]
T8

### 셀 9 · Min-Max 의 약점
BM25 1위가 2위보다 압도적인 질문. Min-Max 는 목록 안의 최댓값이 스케일을 정한다.
- 마지막 표 : 이 질문에서 정답 지문의 순위 — BM25 · 벡터 단독과 `MM_ALPHAS` 의 Min-Max

In [ ]:
# 셀 9 · Min-Max 의 약점
ratio = {qid: runs["bm25"][qid][0][1] / max(runs["bm25"][qid][1][1], 1e-9) for qid in QIDS if len(runs["bm25"][qid]) >= TOP}
qx = max(ratio, key=ratio.get)
sc = np.array([s for _, s in runs["bm25"][qx]])
fig, axes = plt.subplots(1, 2, figsize=(14, 3.6))
axes[0].bar(range(1, len(sc) + 1), sc, color="#3D484A"); axes[0].set_title(f"BM25 점수: 1위가 2위의 {ratio[qx]:.1f}배")
axes[1].bar(range(1, len(sc) + 1), (sc - sc.min()) / (sc.max() - sc.min()), color="#FF7043")
axes[1].set_title("Min-Max 후: 2위부터 0 근처로 눌린다")
for a in axes:
    a.set_xlabel("순위")
plt.tight_layout(); plt.show()
print("질문:", runs["question"][qx])


def gold_rank(lst, qid):
    lst = list(lst); g = runs["gold"][qid]
    return lst.index(g) + 1 if g in lst else "50위 밖"


ranks9 = {"BM25": gold_rank([d for d, _ in runs["bm25"][qx]], qx), "벡터": gold_rank([d for d, _ in runs["vec"][qx]], qx)}
ranks9.update({f"Min-Max α={a}": gold_rank(minmax_fuse("bm25", "vec", a)[qx], qx) for a in MM_ALPHAS})
pd.DataFrame([ranks9], index=["정답 순위"])

### 셀 10 · 종합

In [ ]:
# 셀 10 · 종합
best_mm = mm.loc[mm["MRR"].idxmax()]
best_w = wrrf.loc[wrrf["MRR"].idxmax()]
summary = pd.DataFrame({
    "BM25": single["bm25"], "벡터": single["vec"],
    "RRF k=60": evaluate(rrf(["bm25", "vec"], 60)),
    f"RRF 최고 (k={int(best_rrf['k'])})": best_rrf[["Hit@10", "MRR"]].to_dict(),
    f"가중 RRF (BM25×{best_w['BM25 가중치']})": best_w[["Hit@10", "MRR"]].to_dict(),
    **{f"Min-Max α={a}": mm[np.isclose(mm["α (BM25 비중)"], a)].iloc[0][["Hit@10", "MRR"]].to_dict() for a in MM_ALPHAS},
}).T.astype(float)
if not np.isclose(best_mm["α (BM25 비중)"], MM_ALPHAS).any():          # 최고 α 가 MM_ALPHAS 밖이면 한 줄 더
    summary.loc[f"Min-Max 최고 (α={best_mm['α (BM25 비중)']})"] = best_mm[["Hit@10", "MRR"]].astype(float).values
fig, ax = plt.subplots(figsize=(12, 0.6 * len(summary) + 1))
y = np.arange(len(summary))
ax.barh(y + 0.2, summary["Hit@10"], 0.4, color="#3D484A", label="Hit@10"); ax.barh(y - 0.2, summary["MRR"], 0.4, color="#FF7043", label="MRR")
for i, (h, m) in enumerate(summary.values):
    ax.text(h, i + 0.2, f" {h:.3f}", va="center", fontsize=8); ax.text(m, i - 0.2, f" {m:.3f}", va="center", fontsize=8)
ax.set_yticks(y, summary.index); ax.invert_yaxis(); ax.set_xlim(0, 1.1)
ax.legend(loc="upper center", bbox_to_anchor=(0.5, -0.06), ncol=2)      # 범례는 그림 밖 아래 — 막대 값을 가리지 않게
ax.set_title(f"질문 {len(QIDS)}개 종합"); plt.tight_layout(); plt.show()
summary.round(3)